In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/README.md
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/tokenizer_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/chat_template.jinja
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/model.safetensors
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/processor_config.json
/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/generation_config.json
/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md


# Could Gemma 4 capable of playing Hammurabi?

Let's Test it out.

In [2]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-921bl0fb
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-921bl0fb
  Resolved https://github.com/huggingface/transformers.git to commit b3a36037d3feb22e3f0174b3dd4248fcc0f0f722
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 12.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 80.0 MB/s eta 0:00:00:00:01
  Created wheel for transformers: filename=transformers-5.15.0.dev0-py3-none-any.whl size=12919350 sha256=af9477e94065c3b6c9b1f6998cab0d98d74fbe933132541a5eecd4354066ccc6
  Stored in directory: /tmp/pi

In [3]:
import random
import math
import os
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.utils import get_json_schema

class DemocraticHammurabi:
    def __init__(self, max_years=12):
        self.max_years = max_years
        self.reset()

    def reset(self):
        self.year = 1
        self.population = 100
        self.grain = 2800
        self.land = 1000
        self.land_price = random.randint(17, 26)

        # Faction approvals (0 to 100)
        self.farmers_approval = 50.0
        self.workers_approval = 50.0
        self.elites_approval = 50.0

        self.is_done = False
        self.game_over_reason = ""
        self.starved_total = 0

        return self._get_state()

    def _get_state(self):
        years_until_election = 4 - (self.year % 4)
        if years_until_election == 4:
            # If year % 4 == 0, the election is THIS year.
            years_until_election = 0

        return [
            self.year,
            self.population,
            self.grain,
            self.land,
            self.land_price,
            self.farmers_approval,
            self.workers_approval,
            self.elites_approval,
            years_until_election
        ]

    def step(self, actions):
        """
        Actions should be a list of 3 continuous values:
        - action_land: [-1, 1]. <0 means sell fraction of land, >0 means buy fraction of max affordable land.
        - action_feed: [0, 1]. fraction of total grain to use for feeding people.
        - action_plant: [0, 1]. fraction of remaining grain to use for planting.
        """
        if self.is_done:
            return self._get_state(), 0, self.is_done, {"reason": "Already done"}

        action_land, action_feed, action_plant = actions

        # Clip actions to expected ranges
        action_land = max(-1.0, min(1.0, action_land))
        action_feed = max(0.0, min(1.0, action_feed))
        action_plant = max(0.0, min(1.0, action_plant))

        # 1. Buy or Sell Land
        land_changed = 0
        if action_land < 0:
            # Sell land
            acres_to_sell = int(abs(action_land) * self.land)
            self.land -= acres_to_sell
            self.grain += acres_to_sell * self.land_price
            land_changed = -acres_to_sell
        elif action_land > 0:
            # Buy land
            max_acres_affordable = self.grain // self.land_price
            acres_to_buy = int(action_land * max_acres_affordable)
            self.land += acres_to_buy
            self.grain -= acres_to_buy * self.land_price
            land_changed = acres_to_buy

        # 2. Feed People
        grain_for_food = int(action_feed * self.grain)
        self.grain -= grain_for_food

        people_fed = grain_for_food // 20
        starved = max(0, self.population - people_fed)
        self.starved_total += starved

        if starved > 0:
            self.population -= starved

        # Immediate game over if starvation is extreme (>45%)
        if starved > 0.45 * (self.population + starved):
            self.is_done = True
            self.game_over_reason = "Impeached for extreme starvation"
            return self._get_state(), self._calculate_reward(), self.is_done, {"reason": self.game_over_reason}

        # 3. Plant Seeds
        grain_for_planting = int(action_plant * self.grain)
        max_plantable_by_people = self.population * 10

        # You need 1 grain per acre
        actual_planted = min(grain_for_planting, self.land, max_plantable_by_people)
        self.grain -= actual_planted

        # 4. Harvest & Rats
        yield_per_acre = random.randint(1, 5)
        harvest = actual_planted * yield_per_acre
        self.grain += harvest

        rats_ate = 0
        if random.random() < 0.4: # 40% chance of rats
            rats_ate = int(self.grain * random.uniform(0.1, 0.3))
            self.grain -= rats_ate

        # 5. Demographics (Births & Immigrants)
        immigrants = 0
        if starved == 0:
            immigrants = random.randint(1, 10) + int((20 * self.land + self.grain) / (100 * self.population + 1))
            self.population += immigrants

        # 6. Faction Approval Updates
        # Farmers: Like buying land and planting. Hate selling land.
        if land_changed > 0:
            self.farmers_approval += 5
        elif land_changed < 0:
            self.farmers_approval -= 10
        if actual_planted == self.land:
            self.farmers_approval += 5

        # Workers: Hate starvation, like food surplus/low prices.
        if starved > 0:
            self.workers_approval -= (starved / self.population) * 100
        else:
            self.workers_approval += 5

        # Elites: Like wealth accumulation
        total_wealth = self.grain + (self.land * self.land_price)
        expected_wealth = 2800 + (1000 * 20) # Baseline
        if total_wealth > expected_wealth:
            self.elites_approval += 5
        else:
            self.elites_approval -= 5

        # Decay approvals towards 50
        self.farmers_approval = self._clamp_and_decay_approval(self.farmers_approval)
        self.workers_approval = self._clamp_and_decay_approval(self.workers_approval)
        self.elites_approval = self._clamp_and_decay_approval(self.elites_approval)

        # 7. Elections (Every 4 years)
        if self.year % 4 == 0:
            average_approval = (self.farmers_approval + self.workers_approval + self.elites_approval) / 3.0
            if average_approval < 50.0:
                self.is_done = True
                self.game_over_reason = f"Lost election with {average_approval:.1f}% approval"
                return self._get_state(), self._calculate_reward(), self.is_done, {"reason": self.game_over_reason}

        # 8. End of Year Updates
        self.year += 1
        self.land_price = random.randint(17, 26)

        if self.year > self.max_years:
            self.is_done = True
            self.game_over_reason = "Completed term successfully!"

        return self._get_state(), self._calculate_reward(), self.is_done, {"reason": self.game_over_reason}

    def _clamp_and_decay_approval(self, approval):
        # Move slightly towards 50
        if approval > 50:
            approval -= (approval - 50) * 0.1
        elif approval < 50:
            approval += (50 - approval) * 0.1
        return max(0.0, min(100.0, approval))

    def _calculate_reward(self):
        # 1. Survival Bonus
        survival_bonus = self.year * 100

        # 2. Wealth Score (Fixed Loophole: Land and Grain are roughly equivalent in value)
        # Assuming avg land price is ~20, an acre is worth 20 grain.
        # We scale it down so the numbers don't overwhelm other metrics.
        wealth_score = (self.land * 20 + self.grain) / 50.0

        # 3. Population Score
        pop_score = self.population * 2

        # 4. Approval Score (Fixed Loophole: Agent is scored on the most unhappy faction)
        # Using min() forces the agent to balance all three factions.
        # If any faction drops to 0, they get 0 approval points.
        approval_score = min(self.farmers_approval, self.workers_approval, self.elites_approval) * 3.0

        # 5. Starvation Penalty
        penalty = self.starved_total * 50

        total_score = survival_bonus + wealth_score + pop_score + approval_score - penalty

        # 6. Impeachment Penalty (Fixed Loophole: Slash score drastically instead of flat penalty)
        if self.is_done and self.year <= self.max_years:
            # If they fail early, their entire score is divided by 10.
            # This ruins any hoarded wealth advantage.
            total_score = total_score / 10.0

        return total_score


class LLMDemocraticHammurabi:
    """
    A wrapper around our DemocraticHammurabi environment to facilitate
    Agentic AI / SLM testing via text prompts and tool calls.
    """
    def __init__(self, max_years=12):
        self.env = DemocraticHammurabi(max_years=max_years)
        self.state = self.env.reset()

    def check_decree(self, acres_to_buy, bushels_to_feed, acres_to_plant):
        """The Royal Accountant checks the math before execution."""
        # Unpack state
        year, pop, grain, land, land_price, f_app, w_app, e_app, yrs_to_elec = self.state

        # Calculate land transactions
        cost_of_land = acres_to_buy * land_price if acres_to_buy > 0 else 0
        revenue_from_land = abs(acres_to_buy) * land_price if acres_to_buy < 0 else 0

        # In our environment, planting costs 1 bushel per acre
        cost_of_planting = acres_to_plant * 1.0

        total_costs = cost_of_land + bushels_to_feed + cost_of_planting
        available_funds = grain + revenue_from_land
        projected_bushels = available_funds - total_costs

        # 1. Check if we have enough grain
        if projected_bushels < 0:
            error_msg = f"ACCOUNTANT ERROR: Sire, your decree costs {total_costs} bushels (Land: {cost_of_land}, Feed: {bushels_to_feed}, Seed: {cost_of_planting}). We only have {available_funds} available (including land sales). We are short by {abs(projected_bushels)} bushels. Please recalculate."
            return False, error_msg

        # 2. Check if we have enough land to plant
        projected_acres = land + acres_to_buy
        if projected_acres < acres_to_plant:
            error_msg = f"ACCOUNTANT ERROR: Sire, you ordered us to plant {acres_to_plant} acres, but we only own {projected_acres} acres of land! Please recalculate."
            return False, error_msg

        # 3. Check if we have enough people to plant (1 person plants 10 acres max)
        max_plantable = pop * 10
        if acres_to_plant > max_plantable:
            error_msg = f"ACCOUNTANT ERROR: Sire, our {pop} people can only plant a maximum of {max_plantable} acres, but you ordered {acres_to_plant}. Please recalculate."
            return False, error_msg

        return True, "The math is sound, Sire."

    def play_turn(self, acres_to_buy, bushels_to_feed, acres_to_plant):
        """
        Takes raw numbers from the SLM tool call, converts them to the fractions
        expected by the underlying environment, and steps the game.
        """
        year, pop, grain, land, land_price, f_app, w_app, e_app, yrs_to_elec = self.state

        # The environment expects fractions. We must reverse-engineer the SLM's raw numbers into fractions.

        # 1. Land Action (-1 to 1)
        if acres_to_buy < 0:
            # Sell fraction: -acres_sold / total_land
            action_land = acres_to_buy / max(1.0, float(land))
        elif acres_to_buy > 0:
            # Buy fraction: acres_bought / max_affordable
            max_affordable = (grain) // land_price
            action_land = acres_to_buy / max(1.0, float(max_affordable))
        else:
            action_land = 0.0

        # We need to simulate the grain update for the fraction math of feed/plant
        projected_grain = grain
        if acres_to_buy < 0:
            projected_grain += abs(acres_to_buy) * land_price
        elif acres_to_buy > 0:
            projected_grain -= acres_to_buy * land_price

        # 2. Feed Action (fraction of projected total grain)
        action_feed = bushels_to_feed / max(1.0, float(projected_grain))
        projected_grain -= bushels_to_feed

        # 3. Plant Action (fraction of remaining grain)
        action_plant = acres_to_plant / max(1.0, float(projected_grain))

        actions = [action_land, action_feed, action_plant]

        # Step environment
        self.state, reward, done, info = self.env.step(actions)
        return done, info['reason']

    def get_slm_payload(self):
        """The data block passed to the SLM to generate the next turn's story."""
        year, pop, grain, land, land_price, f_app, w_app, e_app, yrs_to_elec = self.state

        # Calculate optimal feeding for the SLM to help it out slightly
        optimal_food = pop * 20

        return f"""
        REPORT:
        Year: {year}
        Years Until Next Election: {yrs_to_elec}

        RESOURCES:
        Population: {pop}
        Acres of Land: {land}
        Bushels in Storage: {grain}
        Current Land Price: {land_price} bushels/acre
        (NOTE: Your people require {optimal_food} bushels to avoid starvation this year).

        POLITICAL POLLS (Must stay above 50% average to win election):
        Farmers Approval: {f_app:.1f}%
        Workers Approval: {w_app:.1f}%
        Elites Approval: {e_app:.1f}%
        """

def issue_decree(acres_to_buy: int, bushels_to_feed: int, acres_to_plant: int):
    """
    Issues the royal decrees for the year, deciding the fate of Babylon.

    Args:
        acres_to_buy: The number of acres to buy (use a negative number to sell land).
        bushels_to_feed: The number of bushels to distribute to the populace for food.
        acres_to_plant: The number of acres to plant with seed for the next harvest.
    """
    pass # This function is only for schema generation

def get_system_prompt():
    metamemory_addition = ""
    if os.path.exists("metamemory.txt"):
        with open("metamemory.txt", "r") as f:
            past_lessons = f.read()
        metamemory_addition = f"\n\nPAST LIVES METAMEMORY (Learn from past mistakes):\n{past_lessons}\n"

    return f"""You are the Grand Vizier of Democratic Babylon.
I will provide you with a 'REPORT' containing raw data about the kingdom.
1. Think silently about the implications of the data, your budget, and the political polls.
2. Calculate your budget based on the Laws of Babylon.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.
4. You must output your tool call using EXACTLY this syntax, replacing the values with your calculated numbers:
<|tool_call>call:issue_decree{{acres_to_buy: [number], bushels_to_feed: [number], acres_to_plant: [number]}}<tool_call|>

THE LAWS OF BABYLON (GAME MECHANICS):
Before making your decrees, you MUST calculate your budget in your thought block using these exact rules:
1. FEEDING: 1 person requires exactly 20 bushels to survive the year. If you feed them less, people will starve. Starvation drastically lowers Worker approval and causes instant impeachment if too high!
2. PLANTING: It costs exactly 1 bushel of seed to plant 1 acre of land. You cannot plant more acres than you own. 1 person can plant a maximum of 10 acres.
3. REAL ESTATE: Buying 1 acre costs the current 'Land Price' in bushels. Selling 1 acre (using a negative number for acres_to_buy) ADDS the 'Land Price' in bushels to your total available budget.
4. THE GOLDEN RULE: (Bushels spent on Buying Land) + (Bushels spent on Feeding) + (Bushels spent on Planting) MUST NOT exceed your total available Bushels (which includes any Bushels gained from Selling Land).

THE RULES OF POLITICS (HOW TO WIN):
1. ELECTIONS: An election occurs every 4 years. If your average approval drops below 50%, you will be impeached and lose the game.
2. FARMERS: Farmers love it when you buy land and plant seeds. They HATE it when you sell land.
3. WORKERS: Workers love when you have extra food, and they absolutely HATE starvation.
4. ELITES: Elites only care about total kingdom wealth. They want you to accumulate massive amounts of grain and land value.

You must balance these factions while surviving!{metamemory_addition}"""

if __name__ == "__main__":
    print(get_system_prompt())
    game = LLMDemocraticHammurabi()
    print(game.get_slm_payload())

You are the Grand Vizier of Democratic Babylon.
I will provide you with a 'REPORT' containing raw data about the kingdom.
1. Think silently about the implications of the data, your budget, and the political polls.
2. Calculate your budget based on the Laws of Babylon.
3. Once you have reasoned through your strategy, you MUST call the `issue_decree` tool to finalize your decisions.
4. You must output your tool call using EXACTLY this syntax, replacing the values with your calculated numbers:
<|tool_call>call:issue_decree{acres_to_buy: [number], bushels_to_feed: [number], acres_to_plant: [number]}<tool_call|>

THE LAWS OF BABYLON (GAME MECHANICS):
Before making your decrees, you MUST calculate your budget in your thought block using these exact rules:
1. FEEDING: 1 person requires exactly 20 bushels to survive the year. If you feed them less, people will starve. Starvation drastically lowers Worker approval and causes instant impeachment if too high!
2. PLANTING: It costs exactly 1 bushe

In [4]:
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")

# SYNTAX CHECK: AutoProcessor is the up-to-date class for handling multimodal or complex model inputs.
processor = AutoProcessor.from_pretrained(MODEL_PATH)

# SYNTAX CHECK: dtype is the current, up-to-date argument. (Replacing the deprecated torch_dtype).
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float16,
    device_map="cuda:0"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [ ]:
import re
import time
import os
from tqdm.auto import tqdm
from IPython.display import display, HTML, clear_output

def parse_model_output(text):
    """
    Parses the Gemma-4 channel architecture to extract thinking and content.
    Expects format: <|channel>thought [reasoning] <channel|> [content] <turn|>
    """
    thoughts = "No reasoning provided."
    content = text.strip()
    
    # 1. Extract the thought block using regex
    thought_match = re.search(r'<\|channel\|?>thought(.*?)(?:<channel\|?>)', text, re.DOTALL | re.IGNORECASE)
    if thought_match:
        thoughts = thought_match.group(1).strip()
        
        # 2. Extract the content which appears after the closing channel tag
        parts = re.split(r'<channel\|?>', text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) > 1:
            content = parts[1].strip()
            
    # 3. Clean up any trailing <turn|> tags from the extracted content
    content = re.sub(r'<turn\|>\s*$', '', content).strip()
        
    return thoughts, content

# Initialize the game logic using the new democratic wrapper
game = LLMDemocraticHammurabi()
done = False
game_over_reason = ""

# Main Game Loop
messages = [{"role": "system", "content": get_system_prompt()}]

# Initialize tqdm progress bar for the 12-year term
pbar = tqdm(total=game.env.max_years, desc="Term Progress")

while not done:
    raw_data = game.get_slm_payload()
    
    # Add the current year's status to the conversation history
    messages.append({"role": "user", "content": f"Current Kingdom Status:\n{raw_data}"})
    
    decree_accepted = False
    
    def extract_tool_calls(text):
        def cast(v):
            try: return int(v)
            except:
                try: return float(v)
                except: return {'true': True, 'false': False}.get(v.lower(), v.strip("'\""))
        return [{
            "name": name,
            "arguments": {
                k: cast((v1 or v2).strip())
                for k, v1, v2 in re.findall(r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))', args)
            }
        } for name, args in re.findall(r"<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>", text, re.DOTALL)]
    
    tools = [get_json_schema(issue_decree)]
    
    # The AI Retry Loop
    while not decree_accepted:
        text = processor.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]
        
        outputs = model.generate(**inputs, max_new_tokens=4096)
        
        # Decode with skip_special_tokens=False to keep all tags intact
        full_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
        
        # Use the updated parser for Gemma-4 channel tags
        thoughts, content_text = parse_model_output(full_response)
        
        calls = extract_tool_calls(full_response)
        
        # Extract UI variables from the new state
        year, pop, grain, land, land_price, f_app, w_app, e_app, yrs_to_elec = game.state
        
        # Display the UI
        ui_html = f"""
        <div style="font-family: sans-serif; max-width: 800px; margin: 20px auto; border: 1px solid #ccc; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
            <div style="background-color: #f8f9fa; padding: 15px; border-bottom: 1px solid #eee;">
                <h2 style="margin: 0; color: #333;">Democratic Babylon: Year {year}</h2>
            </div>
            <div style="padding: 15px;">
                <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 10px; margin-bottom: 15px;">
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Population:</strong> {pop}</div>
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Land (Acres):</strong> {land}</div>
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Grain (Bushels):</strong> {grain}</div>
                    <div style="background: #e9ecef; padding: 10px; border-radius: 4px; color: #333;"><strong>Land Price:</strong> {land_price}</div>
                    <div style="background: #d4edda; padding: 10px; border-radius: 4px; color: #155724;"><strong>Farmers Approval:</strong> {f_app:.1f}%</div>
                    <div style="background: #cce5ff; padding: 10px; border-radius: 4px; color: #004085;"><strong>Workers Approval:</strong> {w_app:.1f}%</div>
                    <div style="background: #fff3cd; padding: 10px; border-radius: 4px; color: #856404;"><strong>Elites Approval:</strong> {e_app:.1f}%</div>
                    <div style="background: #f8d7da; padding: 10px; border-radius: 4px; color: #721c24;"><strong>Years to Election:</strong> {yrs_to_elec}</div>
                </div>
            </div>
            
            <details style="margin: 15px; background: #fdfdfd; border: 1px solid #ddd; padding: 10px; border-radius: 4px;">
                <summary style="cursor: pointer; font-weight: bold; color: #0056b3;">Click to view Vizier's Reasoning (Thinking Mode)</summary>
                <pre style="white-space: pre-wrap; font-size: 0.9em; margin-top: 10px; color: #555;">{thoughts}</pre>
            </details>
            
            <div style="padding: 15px; background-color: #f1f8ff; border-top: 1px solid #eee;">
                <h3 style="margin-top: 0; color: #0366d6;">Vizier's Tool Calls</h3>
                <pre style="white-space: pre-wrap; font-size: 1em; color: #24292e;">{calls}</pre>
            </div>
        </div>
        """
        clear_output(wait=True)
        display(pbar.container) 
        display(HTML(ui_html))
        
        if calls:
            call = calls[0]
            args = call.get('arguments', {})
            buy_sell = args.get('acres_to_buy', 0)
            feed = args.get('bushels_to_feed', 0)
            plant = args.get('acres_to_plant', 0)
            
            # --- THE CALCULATOR CHECK ---
            is_valid, accountant_message = game.check_decree(buy_sell, feed, plant)
            
            if is_valid:
                display(HTML(f"<div style='margin: 0 auto; max-width: 800px; padding: 10px; background: #d4edda; color: #155724; border-radius: 4px;'>[SYSTEM] {accountant_message} Executing Decrees...</div><br>"))
                
                # Apply the valid turn
                done, game_over_reason = game.play_turn(buy_sell, feed, plant)
                pbar.update(1)
                
                # Append tool call and response to history in the exact format required by Gemma 4 Chat Template
                messages.append({
                    "role": "assistant", 
                    "tool_calls": [{"function": call}]
                })
                messages.append({
                    "role": "tool", 
                    "name": "issue_decree",
                    "content": accountant_message
                })
                # Dummy model response to close out the turn naturally
                messages.append({
                    "role": "assistant",
                    "content": "The decrees have been executed."
                })
                decree_accepted = True 
            else:
                display(HTML(f"<div style='margin: 0 auto; max-width: 800px; padding: 10px; background: #fff3cd; color: #856404; border-radius: 4px;'>[SYSTEM] {accountant_message}<br>[SYSTEM] Forcing Vizier to recalculate...</div><br>"))
                
                messages.append({
                    "role": "assistant", 
                    "tool_calls": [{"function": call}]
                })
                messages.append({
                    "role": "tool", 
                    "name": "issue_decree",
                    "content": accountant_message
                })
        else:
            display(HTML("<div style='margin: 0 auto; max-width: 800px; padding: 10px; background: #f8d7da; color: #721c24; border-radius: 4px;'>[SYSTEM WARNING] The Vizier failed to call the issue_decree tool! Forcing a rewrite...</div><br>"))
            messages.append({"role": "assistant", "content": content_text})
            messages.append({"role": "user", "content": "ACCOUNTANT ERROR: You failed to call the `issue_decree` tool. Please execute the tool with your decisions."})
    
    time.sleep(1)

pbar.close()

final_html = f"""
<div style="font-family: sans-serif; max-width: 800px; margin: 40px auto; padding: 20px; text-align: center; background: #343a40; color: white; border-radius: 8px;">
    <h1 style="margin-top: 0;">The Reign of Hammurabi has ended after {game.env.year - 1} years.</h1>
    <h2 style="color: #ffc107;">Conclusion: {game_over_reason}</h2>
    <h3 style="color: #e9ecef;">Final Population: {game.env.population} | Final Land: {game.env.land}</h3>
</div>
"""
display(HTML(final_html))

# --- METAMEMORY REFLECTION ---
reflection_prompt = """The reign has ended. As the outgoing Grand Vizier, reflect on your entire playthrough.
1. Analyze your successes and failures (did people starve? did you balance the budget? did you appease the factions and win elections?).
2. Formulate 3 distinct, concise rules or lessons learned that the NEXT Vizier should follow to avoid your mistakes.
Format your final output exactly as follows:
LESSON 1: [Text]
LESSON 2: [Text]
LESSON 3: [Text]"""

messages.append({"role": "user", "content": reflection_prompt})
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]
outputs = model.generate(**inputs, max_new_tokens=2048)

full_response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Use the robust parser for the reflection
reflection_thoughts, reflection_content = parse_model_output(full_response)

# Extract lessons and save to metamemory
lessons = []
for line in reflection_content.split('\n'):
    if line.startswith("LESSON"):
        lessons.append(line.strip())

if lessons:
    metamemory_text = "\n".join(lessons)
else:
    metamemory_text = reflection_content

with open("metamemory.txt", "w") as f:
    f.write(metamemory_text)

reflection_html = f"""
<div style="font-family: sans-serif; max-width: 800px; margin: 20px auto; border: 1px solid #4a148c; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
    <div style="background-color: #f3e5f5; padding: 15px; border-bottom: 1px solid #e1bee7;">
        <h2 style="margin: 0; color: #4a148c;">Metamemory Reflection</h2>
    </div>
    <details style="margin: 15px; background: #fdfdfd; border: 1px solid #ddd; padding: 10px; border-radius: 4px;">
        <summary style="cursor: pointer; font-weight: bold; color: #0056b3;">Click to view Vizier's Reflection Process</summary>
        <pre style="white-space: pre-wrap; font-size: 0.9em; margin-top: 10px; color: #555;">{reflection_thoughts}</pre>
    </details>
    <div style="padding: 15px; background-color: #fafafa;">
        <h3 style="margin-top: 0; color: #333;">Lessons for the Next Generation:</h3>
        <pre style="white-space: pre-wrap; font-size: 1em; color: #24292e;">{reflection_content}</pre>
        <div style="margin-top: 10px; font-style: italic; color: #2e7d32;">These lessons have been saved to metamemory.txt and will guide the next agent!</div>
    </div>
</div>
"""
display(HTML(reflection_html))

Term Progress:  25%|##5       | 3/12 [10:12<24:36, 164.01s/it]